<a href="https://colab.research.google.com/github/famub/2026-GP1-02/blob/main/model/KL_leakage_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data-leakage audit: is any patient split across train / val / test?

**The question:** every patient has two knee X-rays (left and right). If one knee
is in training and the other is in validation or test, the model can learn to
recognise the *patient* (bone shape, scanner, disease stage) instead of the
*disease*, and the test score comes out too high. This is the leakage the paper
says inflates other studies' results.

**What this notebook checks**
1. Does any patient appear in more than one set?  (patient overlap)
2. Are the two knees of every patient always together?  (knees split apart)
3. Is the same image file hiding in two different sets?  (exact duplicates)

It runs the same checks on two splits, so we can compare them:
- **A.** the split that ships with the Kaggle dataset (`train/`, `val/`, `test/` folders)
- **B.** our patient-wise split (same code as the main notebook, seed 42)

## 1. Setup

**Does:** installs `kagglehub` (to download the dataset) and imports the libraries.

In [1]:
!pip install -q kagglehub

In [2]:
import os
import hashlib
from pathlib import Path

import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split

## 2. Build the image table

**Does:** makes one row per image: its file path, KL grade, patient ID, which
knee it is, and which set the Kaggle folder puts it in.

**How:** file names look like `9001400L` = patient `9001400`, left knee. The
patient ID is the name without its last letter. We only use `train`, `val` and
`test` (8,260 images, 4,130 patients); the `auto_test` folder is skipped, as in
the main notebook.

In [3]:
root = kagglehub.dataset_download("shashwatwork/knee-osteoarthritis-dataset-with-severity")

rows = []
for folder in ["train", "val", "test"]:
    for grade in ["0", "1", "2", "3", "4"]:
        for path in Path(root).glob(f"**/{folder}/{grade}/*.png"):
            rows.append({
                "path": str(path),
                "kl": int(grade),
                "kaggle_split": folder,          # the split the Kaggle authors made
                "patient_id": path.stem[:-1],    # drop the last letter (L or R)
                "side": path.stem[-1].upper(),
            })

df = pd.DataFrame(rows)
print("images:", len(df), "| patients:", df["patient_id"].nunique())
print("names not ending in L/R:", int((~df["side"].isin(["L", "R"])).sum()))
df.head()

Using Colab cache for faster access to the 'knee-osteoarthritis-dataset-with-severity' dataset.
images: 8260 | patients: 4130
names not ending in L/R: 0


,path,kl,kaggle_split,patient_id,side
0,/kaggle/input/knee-osteoarthritis-dataset-with...,0,train,9641381,R
1,/kaggle/input/knee-osteoarthritis-dataset-with...,0,train,9865771,R
2,/kaggle/input/knee-osteoarthritis-dataset-with...,0,train,9133694,L
3,/kaggle/input/knee-osteoarthritis-dataset-with...,0,train,9112976,L
4,/kaggle/input/knee-osteoarthritis-dataset-with...,0,train,9381081,L


## 3. The audit function

**Does:** takes the table and the name of the column that holds the split, and
runs checks 1 and 2. It prints a short report and returns the numbers.

**Reading the report**
- *patients in more than one set* = leaked patients. **Must be 0.**
- *images that belong to a leaked patient* = how many images are affected.
- *patients with both knees together* should equal the number of patients that
  have two images.
- Patients that only have one image in the whole dataset cannot be split apart,
  so they are counted separately and are not a problem.

In [4]:
def audit(df, split_col, name):
    """Check one split for patient leakage. Prints a report and returns a dict."""
    sets = ["train", "val", "test"]

    # which sets does each patient appear in?
    sets_per_patient = df.groupby("patient_id")[split_col].agg(lambda s: sorted(set(s)))
    n_sets = sets_per_patient.str.len()

    leaked = n_sets[n_sets > 1].index                      # patients in 2+ sets
    leaked_images = int(df["patient_id"].isin(leaked).sum())

    # patients that have two knees in the data
    knees_per_patient = df.groupby("patient_id")["side"].nunique()
    two_knee_patients = knees_per_patient[knees_per_patient == 2].index
    together = n_sets.loc[two_knee_patients].eq(1).sum()    # both knees in one set
    apart = len(two_knee_patients) - int(together)          # knees in different sets

    # pairwise overlap between the sets
    ids = {s: set(df.loc[df[split_col] == s, "patient_id"]) for s in sets}
    overlap = {f"{a}&{b}": len(ids[a] & ids[b])
               for i, a in enumerate(sets) for b in sets[i + 1:]}

    print("=" * 60)
    print(name)
    print("=" * 60)
    print(df.groupby(split_col)["patient_id"].agg(images="size", patients="nunique").loc[sets])
    print()
    print("patients in more than one set:        ", len(leaked))
    print("images that belong to a leaked patient:", leaked_images,
          f"({leaked_images / len(df):.1%} of all images)")
    print("patients with both knees together:    ", int(together), "of", len(two_knee_patients))
    print("patients with knees in different sets:", apart)
    print("patients with only one knee in total: ", int((knees_per_patient == 1).sum()))
    print("overlap between sets (patients):      ", overlap)

    if len(leaked):
        print("\nfirst leaked patients and the sets they appear in:")
        print(sets_per_patient.loc[leaked[:5]].to_string())

    print("\nVERDICT:", "PASS - no patient is split across sets" if len(leaked) == 0
          else f"FAIL - {len(leaked)} patients are split across sets")

    return {"split": name, "leaked_patients": len(leaked), "leaked_images": leaked_images,
            "knees_apart": apart, **{f"overlap_{k}": v for k, v in overlap.items()}}

## 4. Split A: the Kaggle folders

**Does:** audits the `train/` `val/` `test/` split that comes with the dataset.

**Expect:** that split was made image by image, not patient by patient. If so,
the report should show patients whose two knees ended up in different sets.
Whatever it shows is the real answer for this copy of the data.

In [5]:
result_a = audit(df, "kaggle_split", "A. Kaggle folders (train / val / test)")

A. Kaggle folders (train / val / test)
              images  patients
kaggle_split                  
train           5778      2889
val              826       413
test            1656       828

patients in more than one set:         0
images that belong to a leaked patient: 0 (0.0% of all images)
patients with both knees together:     4130 of 4130
patients with knees in different sets: 0
patients with only one knee in total:  0
overlap between sets (patients):       {'train&val': 0, 'train&test': 0, 'val&test': 0}

VERDICT: PASS - no patient is split across sets


## 5. Split B: our patient-wise split

**Does:** rebuilds the split from the main notebook and audits it.

**How (paper, Section 2.1):** label each patient by their highest KL grade, split
the *patients* 70 / 15 / 15 with stratification and seed 42, and send every image
of a patient to that patient's set. This is the same code as step 6 of the main
notebook, so it should give the identical split.

In [6]:
patients = df.groupby("patient_id")["kl"].max()

train_ids, rest_ids = train_test_split(
    patients.index, test_size=0.30, stratify=patients, random_state=42)
val_ids, test_ids = train_test_split(
    rest_ids, test_size=0.50, stratify=patients[rest_ids], random_state=42)

df["patient_split"] = "train"
df.loc[df["patient_id"].isin(val_ids), "patient_split"] = "val"
df.loc[df["patient_id"].isin(test_ids), "patient_split"] = "test"

result_b = audit(df, "patient_split", "B. Patient-wise split (seed 42)")

B. Patient-wise split (seed 42)
               images  patients
patient_split                  
train            5782      2891
val              1238       619
test             1240       620

patients in more than one set:         0
images that belong to a leaked patient: 0 (0.0% of all images)
patients with both knees together:     4130 of 4130
patients with knees in different sets: 0
patients with only one knee in total:  0
overlap between sets (patients):       {'train&val': 0, 'train&test': 0, 'val&test': 0}

VERDICT: PASS - no patient is split across sets


## 6. Check the team's saved `splits.csv` (optional)

**Does:** if the main notebook already saved `splits.csv` to Drive, audits that
file too and confirms it is identical to split B. This is the file everyone on
the team actually trains with, so it is the one that matters.

**How to use:** mount Drive first (`from google.colab import drive;
drive.mount("/content/drive")`), and set `SPLITS_CSV` to the file's path. If the
file is not found, this cell just says so.

In [7]:
SPLITS_CSV = "/content/drive/MyDrive/MUFSAL/run1/splits.csv"

if os.path.exists(SPLITS_CSV):
    saved = pd.read_csv(SPLITS_CSV, dtype={"patient_id": str})
    result_c = audit(saved, "split", "C. Team's saved splits.csv")

    # same image -> same set as split B?
    same = saved.merge(df[["path", "patient_split"]], on="path")
    print("\nimages whose set differs from split B:", int((same["split"] != same["patient_split"]).sum()))
else:
    result_c = None
    print("splits.csv not found at", SPLITS_CSV, "- skipping (mount Drive or fix the path)")

splits.csv not found at /content/drive/MyDrive/MUFSAL/run1/splits.csv - skipping (mount Drive or fix the path)


## 7. Exact duplicate images across sets

**Does:** looks for the same picture stored under two different file names and
sitting in different sets. A duplicate is leakage even when patient IDs look
clean.

**How:** reads every file and computes its MD5 fingerprint (a short code that is
identical only if the files are byte-for-byte the same). The left and right
knees of one patient are different pictures, so they do not count. This reads
8,260 small files and takes a minute or two.

In [8]:
def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

df["hash"] = df["path"].map(file_hash)

def duplicates_across_sets(df, split_col):
    """Image fingerprints that appear in more than one set."""
    sets_per_hash = df.groupby("hash")[split_col].nunique()
    return sets_per_hash[sets_per_hash > 1].index

for col, name in [("kaggle_split", "A. Kaggle folders"), ("patient_split", "B. Patient-wise")]:
    dup = duplicates_across_sets(df, col)
    print(f"{name}: {len(dup)} images appear in more than one set")

print("duplicates inside the whole dataset (any set):",
      int(df["hash"].duplicated().sum()))

A. Kaggle folders: 0 images appear in more than one set
B. Patient-wise: 0 images appear in more than one set
duplicates inside the whole dataset (any set): 0


## 8. Summary

**Does:** puts the splits side by side. The patient-wise split (B), and the
team's `splits.csv` (C) if it was checked, must both show 0 leaked patients
before we trust any test number from the main notebook.

In [9]:
results = [r for r in [result_a, result_b, result_c] if r is not None]
summary = pd.DataFrame(results).set_index("split")
print(summary.T.to_string())

# A hard stop for the split we train on: fail loudly if it leaks.
assert result_b["leaked_patients"] == 0, "patient-wise split leaks!"
if result_c is not None:
    assert result_c["leaked_patients"] == 0, "saved splits.csv leaks!"
print("\nOK: the patient-wise split has no leakage.")

split               A. Kaggle folders (train / val / test)  B. Patient-wise split (seed 42)
leaked_patients                                          0                                0
leaked_images                                            0                                0
knees_apart                                              0                                0
overlap_train&val                                        0                                0
overlap_train&test                                       0                                0
overlap_val&test                                         0                                0

OK: the patient-wise split has no leakage.
